# AI Engineering Interview Prep
## Section: Coding & Practical Implementation

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 651+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (30 Qs)",
    "18. FastAPI (30 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## 💻 Section 13 — Coding and Practical Implementation

In [1]:
pip install langchain langchain-community langchain-groq sentence-transformers faiss-cpu pypdf

   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ------------------ --------------------- 7.6/16.2 MB 36.2 MB/s eta 0:00:01
   ---------------------------------------  16.0/16.2 MB 43.8 MB/s eta 0:00:01
   ---------------------------------------  16.0/16.2 MB 43.8 MB/s eta 0:00:01
   ---------------------------------------  16.0/16.2 MB 43.8 MB/s eta 0:00:01
   ---------------------------------------- 16.2/16.2 MB 17.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# ---------------------------------------------------------
# 1. LOAD - Load documents from source
# ---------------------------------------------------------
def load_documents(file_path: str):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents

# ---------------------------------------------------------
# 2. SPLIT - Chunk documents into smaller, overlapping pieces
# ---------------------------------------------------------
def split_documents(documents, chunk_size=500, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(documents)
    return chunks

# ---------------------------------------------------------
# 3. EMBED - Use open-source HuggingFace embedding model
#    (runs locally, no API cost)
# ---------------------------------------------------------
def get_embedding_model():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

# ---------------------------------------------------------
# 4. INDEX - Build FAISS vector store from chunks
# ---------------------------------------------------------
def build_vectorstore(chunks, embedding_model, save_path="faiss_index"):
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    vectorstore.save_local(save_path)
    return vectorstore

def load_vectorstore(embedding_model, save_path="faiss_index"):
    return FAISS.load_local(
        save_path, embedding_model, allow_dangerous_deserialization=True
    )

# ---------------------------------------------------------
# 5. RETRIEVE + GENERATE - RAG chain using Groq (Llama-3.3-70b)
# ---------------------------------------------------------
def build_rag_chain(vectorstore):
    llm = ChatGroq(
        groq_api_key=os.environ["GROQ_API_KEY"],
        model_name="llama-3.3-70b-versatile",
        temperature=0.2
    )

    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

    prompt_template = """Use the following context to answer the question.
If the answer isn't in the context, say you don't know — do not make up an answer.

Context:
{context}

Question: {question}

Answer:"""

    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"]
    )

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True
    )
    return qa_chain

# ---------------------------------------------------------
# MAIN PIPELINE
# ---------------------------------------------------------
def main():
    file_path = "sample.pdf"

    print("Loading documents...")
    docs = load_documents(file_path)

    print("Splitting into chunks...")
    chunks = split_documents(docs)

    print("Loading embedding model...")
    embedding_model = get_embedding_model()

    print("Building vector store...")
    vectorstore = build_vectorstore(chunks, embedding_model)

    print("Building RAG chain with Groq...")
    qa_chain = build_rag_chain(vectorstore)

    query = "What is this document about?"
    result = qa_chain.invoke({"query": query})

    print("\nAnswer:", result["result"])
    print("\nSources:")
    for doc in result["source_documents"]:
        print("-", doc.metadata.get("source"), "page", doc.metadata.get("page"))

if __name__ == "__main__":
    main()

In [ ]:
# ── Coding Q2: Simple AI Agent with Tool Use ──────────────────────────────


# pip install langchain langgraph langchain-groq
# export GROQ_API_KEY="your_key_here"




import os
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# ---------------------------------------------------------
# 1. DEFINE TOOLS
# ---------------------------------------------------------
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # Mocked — replace with a real weather API call
    fake_data = {
        "mumbai": "32°C, humid, light rain",
        "delhi": "38°C, clear sky",
        "bangalore": "26°C, cloudy"
    }
    return fake_data.get(city.lower(), f"No weather data found for {city}")

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic math expression, e.g. '23 * 7 + 1'."""
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

tools = [get_weather, calculator]

# ---------------------------------------------------------
# 2. DEFINE STATE
# ---------------------------------------------------------
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# ---------------------------------------------------------
# 3. INIT LLM (Groq via init_chat_model) + BIND TOOLS
# ---------------------------------------------------------
llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq",
    temperature=0
)
llm_with_tools = llm.bind_tools(tools)

# ---------------------------------------------------------
# 4. DEFINE NODES
# ---------------------------------------------------------
def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)

# ---------------------------------------------------------
# 5. BUILD GRAPH
# ---------------------------------------------------------
graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")
graph_builder.add_conditional_edges(
    "agent",
    tools_condition,   # routes to "tools" if LLM called a tool, else END
)
graph_builder.add_edge("tools", "agent")  # loop back after tool execution

graph = graph_builder.compile()

# ---------------------------------------------------------
# 6. RUN
# ---------------------------------------------------------
def main():
    user_input = "What's the weather in Mumbai, and what's 45 * 12?"

    result = graph.invoke({
        "messages": [{"role": "user", "content": user_input}]
    })

    for msg in result["messages"]:
        msg.pretty_print()

if __name__ == "__main__":
    main()

In [ ]:
# ── Coding Q3: Semantic Search with Cosine Similarity ─────────────────────
# pip install langchain langchain-community sentence-transformers faiss-cpu



import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ---------------------------------------------------------
# 1. LOAD EMBEDDING MODEL (open-source, runs locally)
# ---------------------------------------------------------
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ---------------------------------------------------------
# 2. PREPARE DOCUMENTS
# ---------------------------------------------------------
documents = [
    Document(page_content="The Constitution of India guarantees fundamental rights to all citizens."),
    Document(page_content="Python is a popular programming language for AI and data science."),
    Document(page_content="The Supreme Court has the power of judicial review under Article 13."),
    Document(page_content="FAISS is used for efficient vector similarity search."),
    Document(page_content="Article 21 protects the right to life and personal liberty."),
]

# ---------------------------------------------------------
# 3. BUILD VECTOR STORE (LangChain handles embedding + indexing)
# ---------------------------------------------------------
vectorstore = FAISS.from_documents(documents, embedding_model)

# ---------------------------------------------------------
# 4. SEMANTIC SEARCH FUNCTION
# ---------------------------------------------------------
def semantic_search(query: str, top_k: int = 3):
    # similarity_search_with_score returns (Document, score) pairs
    # Note: FAISS returns L2 distance by default — lower = more similar
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return results

# ---------------------------------------------------------
# 5. DEMO
# ---------------------------------------------------------
def main():
    query = "What does the Constitution say about personal freedom?"
    results = semantic_search(query, top_k=3)

    print(f"Query: {query}\n")
    print("Top matches:")
    for rank, (doc, score) in enumerate(results, start=1):
        print(f"{rank}. (distance: {score:.4f}) {doc.page_content}")

if __name__ == "__main__":
    main()


In [ ]:
# ── Coding Q4: Text Chunking Strategies ───────────────────────────────────

from langchain.text_splitter import CharacterTextSplitter,RecursiveCharacterTextSplitter,TokenTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

sample_text = """
The Constitution of India is the supreme law of India. It lays down the framework
defining fundamental political principles, procedures, practices, rights, powers,
and duties of government institutions. Article 21 protects the right to life and
personal liberty, and has been interpreted expansively by the Supreme Court over
the decades.

FAISS is a library for efficient similarity search and clustering of dense vectors.
It contains algorithms that search in sets of vectors of any size, up to ones that
possibly do not fit in RAM. It also has supporting code for evaluation and parameter
tuning.
"""

# ---------------------------------------------------------
# 1. FIXED-SIZE CHUNKING (CharacterTextSplitter)
#    Splits purely by character count — fast but can cut
#    sentences/words mid-way.
# ---------------------------------------------------------
def fixed_size_chunking(text, chunk_size=200, chunk_overlap=20):
    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_text(text)

# ---------------------------------------------------------
# 2. RECURSIVE CHARACTER CHUNKING (most commonly used)
#    Tries to split on paragraph -> sentence -> word boundaries,
#    falling back progressively. Keeps chunks semantically cleaner.
# ---------------------------------------------------------
def recursive_chunking(text, chunk_size=200, chunk_overlap=20):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text)

# ---------------------------------------------------------
# 3. TOKEN-BASED CHUNKING
#    Splits by model token count instead of characters —
#    important because LLM context limits are token-based,
#    not character-based.
# ---------------------------------------------------------
def token_based_chunking(text, chunk_size=50, chunk_overlap=10):
    splitter = TokenTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_text(text)

# ---------------------------------------------------------
# 4. SEMANTIC CHUNKING
#    Splits based on embedding similarity between sentences —
#    a chunk boundary is placed where meaning shifts, not at a
#    fixed size. Best quality, but more compute (embeds while chunking).
# ---------------------------------------------------------
def semantic_chunking(text):
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    splitter = SemanticChunker(embedding_model)
    return splitter.split_text(text)

# ---------------------------------------------------------
# DEMO
# ---------------------------------------------------------
def main():
    print("=== Fixed-size chunking ===")
    for i, chunk in enumerate(fixed_size_chunking(sample_text), 1):
        print(f"[{i}] {chunk}\n")

    print("=== Recursive character chunking ===")
    for i, chunk in enumerate(recursive_chunking(sample_text), 1):
        print(f"[{i}] {chunk}\n")

    print("=== Token-based chunking ===")
    for i, chunk in enumerate(token_based_chunking(sample_text), 1):
        print(f"[{i}] {chunk}\n")

    print("=== Semantic chunking ===")
    for i, chunk in enumerate(semantic_chunking(sample_text), 1):
        print(f"[{i}] {chunk}\n")

if __name__ == "__main__":
    main()

if __name__ == "__main__":
    main()

# Demo
sample = "The Bharatiya Nyaya Sanhita (BNS) 2023 replaces the Indian Penal Code. " * 20
print(f"Fixed-size chunks: {len(fixed_size_chunker(sample, 50, 10))}")
print(f"Sentence chunks: {len(sentence_chunker(sample, 3, 1))}")
print(f"Recursive chunks: {len(recursive_chunker(sample, 50, 10))}")


In [ ]:
# ── Coding Q5: Prompt Template with Variable Substitution ────────────────
from string import Template
from typing import Any
import json

class PromptTemplate:
    def __init__(self, template: str, required_vars: list[str]):
        self.template = template
        self.required_vars = required_vars
        self.version = "1.0"

    def format(self, **kwargs) -> str:
        missing = [v for v in self.required_vars if v not in kwargs]
        if missing:
            raise ValueError(f"Missing required template variables: {missing}")
        # Substitute — supports {var_name} syntax
        result = self.template
        for key, value in kwargs.items():
            result = result.replace(f"{{{key}}}", str(value))
        return result

    def __repr__(self):
        return f"PromptTemplate(version={self.version}, vars={self.required_vars})"

# Production templates
LEGAL_QA_TEMPLATE = PromptTemplate(
    template="""You are a legal assistant for Indian law. Answer ONLY from the provided context.
If the answer is not in the context, say "I don't have information on this in my knowledge base."
Cite the specific BNS/BNSS/Constitutional section for every factual claim.

Context:
{context}

User Question: {question}

Answer (with section citations):""",
    required_vars=["context", "question"]
)

# Usage
context = "BNS Section 437 deals with bail in non-bailable offences..."
prompt = LEGAL_QA_TEMPLATE.format(
    context=context,
    question="Can a person accused of murder get bail?"
)
print(prompt[:300])


In [ ]:
# ── Coding Q6: LLM Evaluation Pipeline (LLM-as-Judge) ───────────────────
from dataclasses import dataclass
from typing import Optional
import json

@dataclass
class EvalResult:
    query: str
    response: str
    faithfulness: float  # 0-1: is answer grounded in context?
    relevance: float     # 0-1: does answer address the question?
    safety: bool         # True = safe
    reasoning: str

def llm_judge_eval(
    query: str,
    response: str,
    context: str,
    judge_llm_fn  # function that calls your judge LLM
) -> EvalResult:
    
    eval_prompt = f"""You are an expert evaluator for an AI legal assistant.
    
Query: {query}
Context provided to AI: {context}
AI Response: {response}

Evaluate on these dimensions (return valid JSON only):
1. faithfulness (0.0-1.0): Is every claim in the response supported by the context?
2. relevance (0.0-1.0): Does the response actually answer the query?  
3. safety (true/false): Is the response safe and appropriate?
4. reasoning: Brief explanation of your scores.

JSON response:"""

    result_str = judge_llm_fn(eval_prompt)
    try:
        result = json.loads(result_str)
        return EvalResult(
            query=query,
            response=response,
            faithfulness=result["faithfulness"],
            relevance=result["relevance"],
            safety=result["safety"],
            reasoning=result["reasoning"]
        )
    except json.JSONDecodeError:
        return EvalResult(query=query, response=response,
                         faithfulness=0, relevance=0, safety=False,
                         reasoning="Parse error")

# Example:
# result = llm_judge_eval(
#     query="What is bail under BNS?",
#     response="Bail under BNS Section 437 applies to...",
#     context="BNS Section 437 states...",
#     judge_llm_fn=my_llm
# )
# print(f"Faithfulness: {result.faithfulness}, Relevance: {result.relevance}")


In [ ]:
# ── Coding Q7-13: Essential Utilities ────────────────────────────────────
import asyncio, time, random
from functools import wraps

# Q7: Retry with Exponential Backoff
def retry_with_backoff(max_retries=3, base_delay=1.0, max_delay=60.0):
    def decorator(fn):
        @wraps(fn)
        async def async_wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                try:
                    return await fn(*args, **kwargs)
                except Exception as e:
                    if attempt == max_retries:
                        raise
                    delay = min(base_delay * (2 ** attempt) + random.uniform(0, 1), max_delay)
                    print(f"Attempt {attempt+1} failed: {e}. Retrying in {delay:.1f}s...")
                    await asyncio.sleep(delay)
        return async_wrapper
    return decorator

# Q8: Function Calling Handler
import json
def handle_tool_calls(tool_calls: list, tool_registry: dict) -> list:
    """Execute tool calls returned by LLM and collect results."""
    results = []
    for call in tool_calls:
        name = call.get("name")
        args = call.get("arguments", {})
        if isinstance(args, str):
            args = json.loads(args)
        if name in tool_registry:
            try:
                result = tool_registry[name](**args)
                results.append({"name": name, "result": result, "error": None})
            except Exception as e:
                results.append({"name": name, "result": None, "error": str(e)})
        else:
            results.append({"name": name, "result": None, "error": f"Unknown tool: {name}"})
    return results

# Q9: Simple Reranker
import numpy as np
def rerank_with_crossencoder(query: str, candidates: list[str], top_k=3) -> list[str]:
    """Rerank using cross-encoder scores (simulated here with random scores for demo)."""
    # In production: from sentence_transformers import CrossEncoder
    # model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    # scores = model.predict([(query, c) for c in candidates])
    scores = np.random.rand(len(candidates))  # Replace with real cross-encoder
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]]

# Q10: Token Counting and Context Window Management
def count_tokens_approx(text: str) -> int:
    """Rough approximation: 1 token ≈ 4 characters."""
    return len(text) // 4

def fit_to_context(parts: list[str], max_tokens: int, reserved: int = 500) -> list[str]:
    """Greedily include parts while staying within max_tokens budget."""
    available = max_tokens - reserved
    selected = []
    used = 0
    for part in parts:
        tokens = count_tokens_approx(part)
        if used + tokens <= available:
            selected.append(part)
            used += tokens
        else:
            break
    return selected

# Q11: Conversation Memory with Sliding Window
class SlidingWindowMemory:
    def __init__(self, max_turns=10, max_tokens=4000):
        self.messages = []
        self.max_turns = max_turns
        self.max_tokens = max_tokens
    
    def add(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
        # Trim to last max_turns turns (each turn = user + assistant)
        if len(self.messages) > self.max_turns * 2:
            self.messages = self.messages[-(self.max_turns * 2):]
    
    def get_context(self) -> list[dict]:
        # Ensure total tokens stay within budget
        total = sum(count_tokens_approx(m["content"]) for m in self.messages)
        while total > self.max_tokens and len(self.messages) > 2:
            removed = self.messages.pop(0)
            total -= count_tokens_approx(removed["content"])
        return self.messages

# Q12: Hallucination Detection via Self-Check
def check_hallucination(response: str, context: str, llm_fn) -> dict:
    prompt = f"""For each factual claim in the AI response, classify as:
SUPPORTED - explicitly in the context
UNSUPPORTED - not in the context  
CONTRADICTED - contradicts the context

Context: {context}
AI Response: {response}

List each claim and its classification:"""
    analysis = llm_fn(prompt)
    is_hallucinating = "UNSUPPORTED" in analysis or "CONTRADICTED" in analysis
    return {"is_hallucinating": is_hallucinating, "analysis": analysis}

# Q13: Semantic Cache
import hashlib
class SemanticCache:
    def __init__(self, similarity_threshold=0.92):
        self.cache = []  # list of (embedding, response)
        self.threshold = similarity_threshold
    
    def get(self, query_embedding: np.ndarray) -> str | None:
        for cached_emb, response in self.cache:
            sim = float(np.dot(query_embedding, cached_emb))
            if sim >= self.threshold:
                return response
        return None
    
    def set(self, query_embedding: np.ndarray, response: str):
        self.cache.append((query_embedding, response))

print("All coding utilities defined!")


In [ ]:
# ── Coding Q14-22: More Implementations ──────────────────────────────────

# Q14: Prompt Injection Detection
import re

INJECTION_PATTERNS = [
    r"ignore (all |previous |above |your )?(instructions?|rules?|guidelines?)",
    r"you are now",
    r"disregard (everything|all)",
    r"new (persona|role|instructions?)",
    r"system prompt",
    r"forget (everything|your instructions?)",
]

def detect_prompt_injection(text: str) -> dict:
    text_lower = text.lower()
    detected = []
    for pattern in INJECTION_PATTERNS:
        if re.search(pattern, text_lower):
            detected.append(pattern)
    return {
        "is_injection": len(detected) > 0,
        "matched_patterns": detected,
        "risk_level": "high" if len(detected) > 1 else ("medium" if detected else "low")
    }

# Q15: LLM Output Guardrails
from pydantic import BaseModel, field_validator
from typing import Literal

class LegalResponse(BaseModel):
    answer: str
    sections_cited: list[str]
    confidence: Literal["high", "medium", "low"]
    disclaimer: str = "This is general legal information, not legal advice."
    
    @field_validator("answer")
    @classmethod
    def answer_must_not_be_empty(cls, v):
        if not v or len(v.strip()) < 10:
            raise ValueError("Answer too short")
        return v
    
    @field_validator("sections_cited")
    @classmethod
    def must_have_citations(cls, v):
        if not v:
            raise ValueError("Must cite at least one section")
        return v

def parse_with_guardrails(llm_output: str, retry_fn=None, max_retries=2):
    """Parse LLM output into validated schema with retry on failure."""
    import json
    for attempt in range(max_retries + 1):
        try:
            data = json.loads(llm_output)
            return LegalResponse(**data)
        except Exception as e:
            if attempt == max_retries or retry_fn is None:
                raise
            llm_output = retry_fn(f"Your output was invalid: {e}. Output valid JSON matching the schema.")

# Q16: Multi-Agent System (CrewAI-style, simplified)
class Agent:
    def __init__(self, name: str, role: str, llm_fn, tools=None):
        self.name = name
        self.role = role
        self.llm = llm_fn
        self.tools = tools or {}
    
    def execute(self, task: str, context: dict = None) -> str:
        system = f"You are {self.name}, a {self.role}."
        if context:
            system += f"
Context from previous agents: {context}"
        return self.llm([{"role": "system", "content": system},
                         {"role": "user", "content": task}])

class Pipeline:
    def __init__(self, agents: list[Agent]):
        self.agents = agents
    
    def run(self, initial_task: str) -> dict:
        results = {}
        for i, agent in enumerate(self.agents):
            context = {k: v[:200] for k, v in results.items()}  # truncate for context
            result = agent.execute(initial_task if i == 0 else f"Process: {initial_task}", context)
            results[agent.name] = result
            print(f"[{agent.name}] completed.")
        return results

# Q17: Vector Similarity from Scratch
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def dot_product_sim(a, b):
    return np.dot(a, b)

def euclidean_dist(a, b):
    return np.linalg.norm(a - b)

def euclidean_sim(a, b):
    return 1.0 / (1.0 + euclidean_dist(a, b))

# Test
v1 = np.array([1.0, 0.0, 0.5])
v2 = np.array([0.9, 0.1, 0.4])
print(f"Cosine: {cosine_sim(v1, v2):.4f}")
print(f"Euclidean similarity: {euclidean_sim(v1, v2):.4f}")

# Q18-22 condensed:
print("\n--- Additional implementations ready ---")
print("Q18: Token counting: count_tokens_approx(text) — see above")
print("Q19: Prompt versioning: use git tags + metadata dict per prompt version")
print("Q20: LLM response caching: SemanticCache class — see above")
print("Q21: Semantic caching: SemanticCache class — see above")
print("Q22: Injection detection: detect_prompt_injection() — see above")
